# MADRL Performance Lab

Standalone sweep notebook for short-run MADRL performance experiments. It leaves `train_madrl_grid.ipynb` unchanged and writes runs under a separate experiment name.


In [ ]:
from pathlib import Path
import sys
import warnings

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'configs').exists():
    project_root = project_root.parent
if not (project_root / 'configs').exists():
    raise RuntimeError('Could not locate the project root.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Python executable: {sys.executable}')
project_root


In [ ]:
import importlib.util
import os
from copy import deepcopy

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import torch
from IPython.display import display

torch.set_num_threads(1)

from configs import compose_experiment_config, recommended_gpu_fast_num_envs
from scripts.plots.reward_plots import plot_reward_decomposition
from scripts.utils.experiment_notebook_utils import get_madrl_checkpoint_root
from scripts.utils.grid_notebook_workflow import (
    apply_notebook_experiment_settings,
    collect_madrl_rollout,
    ensure_forecast_ready,
    plot_test_rollout,
    plot_test_voltage_profile,
)
from scripts.utils.madrl_perf_lab import (
    build_candidate_experiment_name,
    build_perf_leaderboard,
    merge_control_overrides,
    recommend_perf_candidate,
    summarize_rollout_metrics,
)
from scripts.utils.train_mainline_launcher import run_external_train_mainline
from scripts.utils.torch_runtime import configure_torch_runtime, describe_device

warnings.filterwarnings('ignore', message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*')


In [ ]:
experiment_name_base = "grid_mainline_perf_lab"
train_episodes_short = 50
evaluate_after_train = True
enable_compile_candidates = [False, True]

experiment_controls = {
    "algorithm": "MATD3",
    "reward_plot_window": 10,
    "seed": 7,
    "runtime_mode": "performance",
    "device_request": "cuda" if torch.cuda.is_available() else "cpu",
    "require_cuda": bool(torch.cuda.is_available()),
    "model_controls": {
        "hidden_dim": 256,
    },
    "runtime_controls": {
        "enable_amp": True,
        "amp_dtype": "bfloat16",
        "pin_memory": True,
        "non_blocking_transfers": True,
        "enable_compile": True,
        "compile_mode": "reduce-overhead",
        "compile_fullgraph": False,
        "compile_dynamic": False,
    },
}

data_controls = {
    "prediction_mode": "normal",
    "test_start_date": 20200601,
    "test_end_date": 20200610,
    "agent_profiles": ["SFH12", "SFH14", "SFH16"],
    "load_scale": [5.0, 5.0, 5.0],
    "pv_scale": [5.0, 5.0, 5.0],
    "future_horizon": 24,
    "train_year": 2019,
    "test_year": 2020,
}

battery_controls = {
    "mode": "from_pv",
    "from_pv_power_ratio": 0.5,
    "from_pv_duration_hours": 2.0,
    "battery_capacity": 5.0,
    "max_charge_rate": 2.5,
    "efficiency": 0.95,
    "init_soc": 0.5,
    "soc_min": 0.05,
    "soc_max": 0.95,
    "soc_target": 0.5,
}

checkpoint_controls = {
    "experiment_name": experiment_name_base,
    "checkpoint_root": str(get_madrl_checkpoint_root(project_root)),
}

baseline_train_controls = {
    "launch_mode": "external",
    "profile": "gpu_fast",
    "model_family": "mlp",
    "num_envs": recommended_gpu_fast_num_envs(),
    "vec_env_type": "subproc",
    "train_episodes": train_episodes_short,
    "batch_size": 4096,
    "buffer_size": 200000,
    "update_interval": 1,
    "updates_per_step": 2,
    "policy_update_freq": 2,
    "use_noise_decay": True,
    "show_progress": True,
    "progress_postfix_interval": 20,
    "noise_std_init": 0.35,
    "noise_std_min": 0.05,
    "max_train_steps": None,
}

sweep_candidates = {
    "batch_size": [4096, 8192, 12288],
    "updates_per_step": [2, 3, 4],
    "num_envs": [16, 20, 24],
}

compile_supported = bool(torch.cuda.is_available()) and hasattr(torch, 'compile') and importlib.util.find_spec('triton') is not None
reward_plot_window = experiment_controls["reward_plot_window"]
seed = experiment_controls["seed"]
train_env_name = "GridPerfLab"
train_run_number = 1

control_summary = pd.DataFrame([
    {
        "experiment_name_base": experiment_name_base,
        "device_request": experiment_controls["device_request"],
        "train_episodes_short": train_episodes_short,
        "baseline_num_envs": baseline_train_controls["num_envs"],
        "baseline_batch_size": baseline_train_controls["batch_size"],
        "baseline_updates_per_step": baseline_train_controls["updates_per_step"],
        "hidden_dim": experiment_controls["model_controls"]["hidden_dim"],
        "compile_supported": compile_supported,
    }
])
display(control_summary)
if not compile_supported:
    print('compile stage will be skipped because Triton or torch.compile support is unavailable in the active environment.')


In [ ]:
def build_cfg_from_controls(experiment_controls_local, train_controls_local):
    cfg = compose_experiment_config(
        profile=train_controls_local["profile"],
        algorithm=experiment_controls_local["algorithm"],
        model_family=train_controls_local["model_family"],
        vec_env_type=train_controls_local["vec_env_type"],
        data_dir=project_root / 'data',
        device=experiment_controls_local["device_request"],
        runtime_mode=experiment_controls_local["runtime_mode"],
        seed=experiment_controls_local["seed"],
        require_cuda=experiment_controls_local["require_cuda"],
    )
    model_controls = dict(experiment_controls_local.get("model_controls", {}))
    runtime_controls = dict(experiment_controls_local.get("runtime_controls", {}))
    if "hidden_dim" in model_controls:
        cfg.model.hidden_dim = int(model_controls["hidden_dim"])

    applied_controls = apply_notebook_experiment_settings(
        cfg,
        prediction_mode=data_controls["prediction_mode"],
        test_start_date=data_controls["test_start_date"],
        test_end_date=data_controls["test_end_date"],
        agent_profiles=data_controls["agent_profiles"],
        load_scale=data_controls["load_scale"],
        pv_scale=data_controls["pv_scale"],
        battery_controls=battery_controls,
        future_horizon=data_controls["future_horizon"],
        train_year=data_controls["train_year"],
        test_year=data_controls["test_year"],
    )

    cfg.train.train_episodes = int(train_controls_local["train_episodes"])
    cfg.train.num_envs = int(train_controls_local["num_envs"])
    cfg.train.vec_env_type = train_controls_local["vec_env_type"]
    cfg.train.batch_size = int(train_controls_local["batch_size"])
    cfg.train.buffer_size = int(train_controls_local["buffer_size"])
    cfg.train.update_interval = int(train_controls_local["update_interval"])
    cfg.train.updates_per_step = int(train_controls_local["updates_per_step"])
    cfg.train.use_noise_decay = bool(train_controls_local["use_noise_decay"])
    cfg.train.show_progress = bool(train_controls_local["show_progress"])
    cfg.train.progress_postfix_interval = int(train_controls_local["progress_postfix_interval"])
    cfg.train.noise_std_init = float(train_controls_local["noise_std_init"])
    cfg.train.noise_std_min = float(train_controls_local["noise_std_min"])
    cfg.train.max_train_steps = train_controls_local["max_train_steps"]
    cfg.algo.policy_update_freq = int(train_controls_local["policy_update_freq"])
    for field_name in (
        "pin_memory",
        "non_blocking_transfers",
        "enable_amp",
        "enable_compile",
        "compile_fullgraph",
        "compile_dynamic",
        "amp_dtype",
        "compile_mode",
        "matmul_precision",
    ):
        if field_name in runtime_controls:
            setattr(cfg.runtime, field_name, runtime_controls[field_name])
    cfg.train.noise_decay_steps = cfg.train.train_episodes * cfg.env.episode_limit

    runtime_state = configure_torch_runtime(
        cfg,
        device=experiment_controls_local["device_request"],
        seed=experiment_controls_local["seed"],
        require_cuda=experiment_controls_local["require_cuda"],
    )
    forecast_ready = ensure_forecast_ready(cfg)
    return cfg, applied_controls, runtime_state, forecast_ready


def find_result_bundle(result_bundles, candidate_name):
    for bundle in result_bundles:
        if bundle["row"]["candidate_name"] == candidate_name:
            return bundle
    raise KeyError(f"Could not find candidate bundle for {candidate_name}.")


def evaluate_stage(stage_results, baseline_name):
    leaderboard = build_perf_leaderboard([bundle["row"] for bundle in stage_results])
    recommendation = recommend_perf_candidate(leaderboard, baseline_name=baseline_name)
    winner = find_result_bundle(stage_results, recommendation["recommended_candidate"])
    return leaderboard, recommendation, winner


In [ ]:
def run_perf_candidate(
    candidate_name,
    *,
    phase,
    base_experiment_controls=None,
    base_train_controls=None,
    train_overrides=None,
    model_overrides=None,
    runtime_overrides=None,
):
    experiment_controls_local = merge_control_overrides(
        base_experiment_controls or experiment_controls,
        {
            "model_controls": model_overrides or {},
            "runtime_controls": runtime_overrides or {},
        },
    )
    train_patch = {"train_episodes": train_episodes_short}
    train_patch.update(dict(train_overrides or {}))
    train_controls_local = merge_control_overrides(base_train_controls or baseline_train_controls, train_patch)
    checkpoint_controls_local = deepcopy(checkpoint_controls)
    checkpoint_controls_local["experiment_name"] = build_candidate_experiment_name(
        experiment_name_base,
        candidate_name,
    )

    cfg = None
    train_result = {}
    launch_metadata = None
    rollout = None
    try:
        cfg, applied_controls, runtime_state, forecast_ready = build_cfg_from_controls(
            experiment_controls_local,
            train_controls_local,
        )
        launch_metadata = run_external_train_mainline(
            project_root=project_root,
            experiment_controls=experiment_controls_local,
            data_controls=data_controls,
            battery_controls=battery_controls,
            train_controls=train_controls_local,
            checkpoint_controls=checkpoint_controls_local,
            data_dir=project_root / 'data',
            env_name=train_env_name,
            run_number=train_run_number,
            stream_output=True,
        )
        train_result = dict(launch_metadata.get("result") or {})
        perf_summary = dict(train_result.get("perf_summary") or {})

        if evaluate_after_train:
            rollout = collect_madrl_rollout(
                cfg,
                model_root=Path(train_result["model_root"]),
                algorithm=train_result.get("algorithm", cfg.algo.name),
                episode_tag=int(train_result.get("saved_episode_tag", 0)),
                experiment_name=checkpoint_controls_local["experiment_name"],
                checkpoint_root=checkpoint_controls_local["checkpoint_root"],
            )
            rollout_metrics = summarize_rollout_metrics(rollout)
        else:
            rollout_metrics = {}

        row = {
            "phase": phase,
            "candidate_name": str(candidate_name),
            "is_baseline": str(candidate_name) == "baseline",
            "status": "ok",
            "error_message": None,
            "experiment_name": checkpoint_controls_local["experiment_name"],
            "train_episodes": int(train_controls_local["train_episodes"]),
            "batch_size": int(train_controls_local["batch_size"]),
            "updates_per_step": int(train_controls_local["updates_per_step"]),
            "num_envs": int(train_controls_local["num_envs"]),
            "hidden_dim": int(experiment_controls_local.get("model_controls", {}).get("hidden_dim", cfg.model.hidden_dim)),
            "enable_compile_requested": experiment_controls_local.get("runtime_controls", {}).get("enable_compile"),
            "run_label": train_result.get("run_label"),
            "saved_episode_tag": train_result.get("saved_episode_tag"),
            "model_root": train_result.get("model_root"),
            "total_wall_time_s": float(perf_summary.get("total_wall_time_s", train_result.get("elapsed_seconds", float('nan')))),
            "steps_per_sec": float(perf_summary.get("steps_per_sec", float('nan'))),
            "avg_env_ms_per_iter": float(perf_summary.get("avg_env_ms_per_iter", float('nan'))),
            "avg_update_ms_per_call": float(perf_summary.get("avg_update_ms_per_call", float('nan'))),
            "amp_enabled": perf_summary.get("amp_enabled"),
            "compile_enabled": perf_summary.get("compile_enabled"),
            "compile_fallback_reason": perf_summary.get("compile_fallback_reason"),
            "forecast_ready": forecast_ready,
            "device_info": describe_device(runtime_state),
        }
        row.update(rollout_metrics)
    except Exception as exc:
        row = {
            "phase": phase,
            "candidate_name": str(candidate_name),
            "is_baseline": str(candidate_name) == "baseline",
            "status": "failed",
            "error_message": f"{type(exc).__name__}: {exc}",
            "experiment_name": checkpoint_controls_local["experiment_name"],
            "train_episodes": int(train_controls_local["train_episodes"]),
            "batch_size": int(train_controls_local["batch_size"]),
            "updates_per_step": int(train_controls_local["updates_per_step"]),
            "num_envs": int(train_controls_local["num_envs"]),
            "hidden_dim": int(experiment_controls_local.get("model_controls", {}).get("hidden_dim", 256)),
            "enable_compile_requested": experiment_controls_local.get("runtime_controls", {}).get("enable_compile"),
            "run_label": None,
            "saved_episode_tag": None,
            "model_root": None,
            "total_wall_time_s": float('nan'),
            "steps_per_sec": float('nan'),
            "avg_env_ms_per_iter": float('nan'),
            "avg_update_ms_per_call": float('nan'),
            "amp_enabled": None,
            "compile_enabled": None,
            "compile_fallback_reason": None,
        }

    return {
        "row": row,
        "experiment_controls": experiment_controls_local,
        "train_controls": train_controls_local,
        "checkpoint_controls": checkpoint_controls_local,
        "train_result": train_result,
        "launch_metadata": launch_metadata,
        "cfg": cfg,
        "rollout": rollout,
    }


In [ ]:
plan_rows = [{"phase": "baseline", "candidate": "baseline", "details": "current gpu_fast baseline"}]
for batch_size in sweep_candidates["batch_size"]:
    plan_rows.append({"phase": "batch_size", "candidate": f"batch_{batch_size}", "details": f"batch_size={batch_size}"})
for updates_per_step in sweep_candidates["updates_per_step"]:
    plan_rows.append({"phase": "updates_per_step", "candidate": f"updates_{updates_per_step}", "details": f"updates_per_step={updates_per_step}"})
for num_envs in sweep_candidates["num_envs"]:
    plan_rows.append({"phase": "num_envs", "candidate": f"env_{num_envs}", "details": f"num_envs={num_envs}"})
if compile_supported:
    for enable_compile in enable_compile_candidates:
        plan_rows.append({"phase": "compile", "candidate": f"compile_{'on' if enable_compile else 'off'}", "details": f"enable_compile={enable_compile}"})
plan_df = pd.DataFrame(plan_rows)
display(plan_df)


In [ ]:
baseline_result = run_perf_candidate("baseline", phase="baseline")
current_best_result = baseline_result
current_best_reason = "Baseline establishes the starting point for the staged sweep."
display(pd.DataFrame([baseline_result["row"]]))

batch_stage_results = [baseline_result]
for batch_size in sweep_candidates["batch_size"]:
    if int(batch_size) == int(baseline_result["train_controls"]["batch_size"]):
        continue
    batch_stage_results.append(
        run_perf_candidate(
            f"batch_{batch_size}",
            phase="batch_size",
            base_experiment_controls=baseline_result["experiment_controls"],
            base_train_controls=baseline_result["train_controls"],
            train_overrides={"batch_size": int(batch_size)},
        )
    )
batch_stage_leaderboard, batch_stage_recommendation, current_best_result = evaluate_stage(
    batch_stage_results,
    baseline_name=baseline_result["row"]["candidate_name"],
)
current_best_reason = batch_stage_recommendation["reason"]
display(batch_stage_leaderboard)
print(batch_stage_recommendation["reason"])
display(pd.DataFrame([batch_stage_recommendation["row"]]))


In [ ]:
updates_stage_results = [current_best_result]
for updates_per_step in sweep_candidates["updates_per_step"]:
    if int(updates_per_step) == int(current_best_result["train_controls"]["updates_per_step"]):
        continue
    updates_stage_results.append(
        run_perf_candidate(
            f"updates_{updates_per_step}",
            phase="updates_per_step",
            base_experiment_controls=current_best_result["experiment_controls"],
            base_train_controls=current_best_result["train_controls"],
            train_overrides={"updates_per_step": int(updates_per_step)},
        )
    )
updates_stage_leaderboard, updates_stage_recommendation, current_best_result = evaluate_stage(
    updates_stage_results,
    baseline_name=updates_stage_results[0]["row"]["candidate_name"],
)
current_best_reason = updates_stage_recommendation["reason"]
display(updates_stage_leaderboard)
print(updates_stage_recommendation["reason"])
display(pd.DataFrame([updates_stage_recommendation["row"]]))

num_env_stage_results = [current_best_result]
for num_envs in sweep_candidates["num_envs"]:
    if int(num_envs) == int(current_best_result["train_controls"]["num_envs"]):
        continue
    num_env_stage_results.append(
        run_perf_candidate(
            f"env_{num_envs}",
            phase="num_envs",
            base_experiment_controls=current_best_result["experiment_controls"],
            base_train_controls=current_best_result["train_controls"],
            train_overrides={"num_envs": int(num_envs)},
        )
    )
num_env_stage_leaderboard, num_env_stage_recommendation, current_best_result = evaluate_stage(
    num_env_stage_results,
    baseline_name=num_env_stage_results[0]["row"]["candidate_name"],
)
current_best_reason = num_env_stage_recommendation["reason"]
display(num_env_stage_leaderboard)
print(num_env_stage_recommendation["reason"])
display(pd.DataFrame([num_env_stage_recommendation["row"]]))


In [ ]:
if compile_supported:
    compile_stage_results = [current_best_result]
    current_compile_flag = bool(current_best_result["experiment_controls"].get("runtime_controls", {}).get("enable_compile", True))
    for enable_compile in enable_compile_candidates:
        if bool(enable_compile) == current_compile_flag:
            continue
        compile_stage_results.append(
            run_perf_candidate(
                f"compile_{'on' if enable_compile else 'off'}",
                phase="compile",
                base_experiment_controls=current_best_result["experiment_controls"],
                base_train_controls=current_best_result["train_controls"],
                runtime_overrides={"enable_compile": bool(enable_compile)},
            )
        )
    compile_stage_leaderboard, compile_stage_recommendation, current_best_result = evaluate_stage(
        compile_stage_results,
        baseline_name=compile_stage_results[0]["row"]["candidate_name"],
    )
    current_best_reason = compile_stage_recommendation["reason"]
    display(compile_stage_leaderboard)
    print(compile_stage_recommendation["reason"])
    display(pd.DataFrame([compile_stage_recommendation["row"]]))
else:
    compile_stage_results = [current_best_result]
    compile_stage_leaderboard = build_perf_leaderboard([current_best_result["row"]])
    compile_stage_recommendation = {
        "recommended_candidate": current_best_result["row"]["candidate_name"],
        "reason": "Compile stage was skipped because Triton or torch.compile support is unavailable in the active environment.",
        "row": current_best_result["row"],
    }
    current_best_reason = compile_stage_recommendation["reason"]
    display(compile_stage_leaderboard)
    print(compile_stage_recommendation["reason"])

all_result_map = {}
for stage_results in (
    batch_stage_results,
    updates_stage_results,
    num_env_stage_results,
    compile_stage_results,
):
    for bundle in stage_results:
        all_result_map[bundle["row"]["candidate_name"]] = bundle
leaderboard = build_perf_leaderboard([bundle["row"] for bundle in all_result_map.values()])
recommended_result = current_best_result
recommended_summary = pd.DataFrame([recommended_result["row"]])
baseline_row = leaderboard.loc[leaderboard["candidate_name"] == "baseline"].iloc[0]
recommended_row = recommended_result["row"]
print(f"Recommended candidate: {recommended_row['candidate_name']}")
print(current_best_reason)
print(f"Operating cost delta vs baseline: {float(recommended_row['total_operating_cost']) - float(baseline_row['total_operating_cost']):.6f}")
print(f"Wall time delta vs baseline: {float(recommended_row['total_wall_time_s']) - float(baseline_row['total_wall_time_s']):.6f}")
display(leaderboard)
display(recommended_summary)


In [ ]:
print(f"Recommended candidate reward summary: {recommended_result['train_result']['reward_summary_path']}")
plot_reward_decomposition(
    reward_summary=Path(recommended_result["train_result"]["reward_summary_path"]),
    title=f"Reward Decomposition - {recommended_result['row']['candidate_name']}",
    window=reward_plot_window,
);

if recommended_result["rollout"] is not None:
    plot_test_rollout(recommended_result["rollout"]);
    plot_test_voltage_profile(recommended_result["rollout"]);
